# 🎤 RVC (Retrieval-based Voice Conversion) CLI Colab 학습 가이드 (파라미터 조절 가능)

이 노트북은 **Gradio WebUI 없이** Google Colab 환경에서 RVC v2 모델을 학습시키는 파이썬 셀 기반 노트북입니다.
WebUI처럼 **Batch Size, Epoch, Sample Rate, Pitch 알고리즘, GPU 캐싱 여부 등 모든 주요 하이퍼파라미터를 입력 폼(#@param)으로 자유롭게 조절**할 수 있습니다.

### 📌 전체 진행 순서
1. **GPU & 환경 설정**
2. **구글 드라이브 연동 & 필수 사전 학습(Pretrained) 모델 다운로드**
3. **데이터셋 업로드 및 압축 해제**
4. **데이터 전처리 (Audio Splitting & Resampling 파라미터 설정)**
5. **특징(Pitch F0 & HuBERT Feature) 추출 방식 설정**
6. **Filelist 및 Config.json 자동 생성**
7. **PyTorch 모델 학습 (Batch Size, Epochs 등 학습 하이퍼파라미터 조절)**
8. **FAISS Feature Index 생성**
9. **완성된 Weights (.pth) & Index (.index) 구글 드라이브 내보내기**

---
## 1. GPU 및 환경 설정
Colab GPU 연결 상태를 확인하고, RVC 저장소 클론 및 필요한 라이브러리를 설치합니다.

In [ ]:
# 1. GPU 상태 확인
!nvidia-smi

import os
# 2. RVC 레포지토리 클론
if not os.path.exists("/content/Retrieval-based-Voice-Conversion-WebUI"):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/Retrieval-based-Voice-Conversion-WebUI

%cd /content/Retrieval-based-Voice-Conversion-WebUI

# 3. aria2 고속 다운로더 및 필수 패키지 설치
!apt-get update -qq && !apt-get install -y -qq aria2
!pip install -r requirements.txt
!pip install faiss-cpu fairseq praat-parselmouth pyworld torch-directml tensorboard

---
## 2. 구글 드라이브 연동 & 필수 사전 학습 모델 다운로드
학습 결과물 저장 및 데이터셋 로드를 위해 Google Drive를 마운트하고, HuBERT, RMVPE 및 Pretrained Baseline 모델을 다운로드합니다.

In [ ]:
from google.colab import drive
import os

# 1. 구글 드라이브 마운트
drive.mount("/content/drive")

# 2. 디렉토리 구조 생성
!mkdir -p assets/hubert assets/rmvpe assets/pretrained assets/pretrained_v2 assets/indices assets/weights

# 3. HuBERT base 다운로드
if not os.path.exists("assets/hubert/hubert_base.pt"):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt -d assets/hubert -o hubert_base.pt

# 4. RMVPE Pitch Extractor 다운로드
if not os.path.exists("assets/rmvpe/rmvpe.pt"):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt -d assets/rmvpe -o rmvpe.pt

# 5. v2 사전 학습 Baseline 모델 (40k, 48k) 다운로드
for sr in ["40k", "48k"]:
    if not os.path.exists(f"assets/pretrained_v2/f0G{sr}.pth"):
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G{sr}.pth -d assets/pretrained_v2 -o f0G{sr}.pth
    if not os.path.exists(f"assets/pretrained_v2/f0D{sr}.pth"):
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D{sr}.pth -d assets/pretrained_v2 -o f0D{sr}.pth

print("✅ 필수 사전 학습 모델 다운로드 완료!")

---
## 3. 학습 데이터셋 준비
Google Drive 상에 올린 음성 데이터셋 압축 파일 (`dataset.zip`)을 풀어 `dataset` 폴더에 위치시킵니다.

In [ ]:
import os

# 💡 구글 드라이브 내 데이터셋 zip 파일 경로
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"  # @param {type:"string"}
DATASET_DIR = "/content/Retrieval-based-Voice-Conversion-WebUI/dataset"

!mkdir -p {DATASET_DIR}

if os.path.exists(ZIP_PATH):
    !unzip -q -o "{ZIP_PATH}" -d "{DATASET_DIR}"
    print(f"✅ {ZIP_PATH} 압축 해제 완료!")
else:
    print(f"⚠️ {ZIP_PATH} 경로를 찾을 수 없습니다. 경로를 수정하거나 dataset 디렉토리에 음성(.wav, .mp3) 파일을 직접 업로드해 주세요.")

---
## 4. 데이터 전처리 (Audio Splitting & Resampling 설정)
오른쪽 Form 우측 창에서 모델 이름, 샘플레이트, CPU 스레드 수, 자르는 시간 단위(초)를 직접 조정할 수 있습니다.

In [ ]:
# @markdown ### ⚙️ 1. 데이터 전처리 파라미터 설정
EXP_NAME = "my_rvc_model"  # @param {type:"string"} - 모델/실험 이름
SAMPLE_RATE = "40k"  # @param ["40k", "48k", "32k"] - 타겟 샘플레이트
NUM_PROCESSES = 2  # @param {type:"integer"} - CPU 병렬 프로세스 수
PREPROCESS_PER = 3.0  # @param {type:"number"} - 자르는 단위 시간(초)

sr_dict = {"40k": 40000, "48k": 48000, "32k": 32000}
sr_hz = sr_dict[SAMPLE_RATE]
LOG_DIR = f"logs/{EXP_NAME}"

# 데이터 전처리 실행
!python train/preprocess.py dataset {sr_hz} {NUM_PROCESSES} {LOG_DIR} False {PREPROCESS_PER}

print("✅ 데이터 전처리 완료!")

---
## 5. Pitch (F0) 및 HuBERT Feature 추출 방식 설정
Pitch 추출 방식 (RMVPE 또는 PM), Half Precision(float16) 사용 여부를 선택합니다.

In [ ]:
# @markdown ### ⚙️ 2. 특징 추출 파라미터 설정
VERSION = "v2"  # @param ["v2", "v1"] - 모델 버전
F0_METHOD = "rmvpe"  # @param ["rmvpe", "pm"] - Pitch 추출 방식 (RMVPE 권장)
IS_HALF = True  # @param {type:"boolean"} - GPU 반정밀도(FP16) 가속 연산

# 1. Pitch (F0) 추출
if F0_METHOD == "rmvpe":
    !python train/dataset/extract_f0.py cuda 1 0 0 logs/{EXP_NAME} {IS_HALF}
else:
    !python train/dataset/extract_f0.py cpu logs/{EXP_NAME} {NUM_PROCESSES} pm

# 2. HuBERT Feature 추출
!python train/dataset/extract_hubert_feature.py cuda 1 0 0 logs/{EXP_NAME} {VERSION} {IS_HALF}

print("✅ F0 및 HuBERT Feature 추출 완료!")

---
## 6. Filelist.txt 및 Config.json 생성
추출된 파일 매핑 및 `config.json`을 자동으로 구성합니다.

In [ ]:
import os
import json
import random

SPK_ID = 0
IF_F0 = True

now_dir = os.getcwd()
exp_dir = os.path.join(now_dir, "logs", EXP_NAME)
gt_wavs_dir = os.path.join(exp_dir, "0_gt_wavs")
feature_dir = os.path.join(exp_dir, "3_feature768" if VERSION == "v2" else "3_feature256")

gt_names = set([n.rsplit('.', 1)[0] for n in os.listdir(gt_wavs_dir) if n.endswith('.wav')])
feat_names = set([n.rsplit('.', 1)[0] for n in os.listdir(feature_dir) if n.endswith('.npy')])
names = gt_names & feat_names

if IF_F0:
    f0_dir = os.path.join(exp_dir, "2a_f0")
    f0nsf_dir = os.path.join(exp_dir, "2b-f0nsf")
    f0_names = set([n.rsplit('.', 1)[0] for n in os.listdir(f0_dir) if n.endswith('.npy')])
    f0nsf_names = set([n.rsplit('.', 1)[0] for n in os.listdir(f0nsf_dir) if n.endswith('.npy')])
    names = names & f0_names & f0nsf_names

opt = []
for name in sorted(list(names)):
    if IF_F0:
        opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{f0_dir}/{name}.wav.npy|{f0nsf_dir}/{name}.wav.npy|{SPK_ID}")
    else:
        opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{SPK_ID}")

fea_dim = 768 if VERSION == "v2" else 256
for _ in range(2):
    if IF_F0:
        opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{SAMPLE_RATE}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{now_dir}/logs/mute/2a_f0/mute.wav.npy|{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|{SPK_ID}")
    else:
        opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{SAMPLE_RATE}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{SPK_ID}")

random.shuffle(opt)

filelist_path = os.path.join(exp_dir, "filelist.txt")
with open(filelist_path, "w", encoding="utf8") as f:
    f.write("\n".join(opt))

config_src = f"configs/{VERSION}/{SAMPLE_RATE}.json"
config_dst = os.path.join(exp_dir, "config.json")
if os.path.exists(config_src):
    with open(config_src, "r", encoding="utf8") as f:
        config_data = json.load(f)
    with open(config_dst, "w", encoding="utf8") as f:
        json.dump(config_data, f, ensure_ascii=False, indent=4)

print(f"✅ filelist.txt 및 config.json 생성 완료! (학습 샘플 개수: {len(names)}개)")

---
## 7. RVC PyTorch 모델 학습 (하이퍼파라미터 조절)
WebUI에서 설정하던 **Batch Size, Total Epochs, Save Frequency, GPU Cache, Baseline Pretrained Model 지정** 등의 하이퍼파라미터를 입력 폼에서 직접 조절할 수 있습니다.

In [ ]:
# @markdown ### ⚙️ 3. 학습 하이퍼파라미터 (Batch Size, Epoch 등) 조절
BATCH_SIZE = 8  # @param {type:"integer"} - 배치 사이즈 (Colab Free T4: 8~16 추천)
TOTAL_EPOCH = 100  # @param {type:"integer"} - 총 학습 에포크 수 (100~300 권장)
SAVE_EPOCH = 20  # @param {type:"integer"} - 체크포인트 저장 주기(에포크)
IF_F0 = True  # @param {type:"boolean"} - 피치(F0) 가이드 사용 여부
IF_CACHE_GPU = True  # @param {type:"boolean"} - 데이터셋 GPU VRAM 캐싱 (학습 속도 향상)
SAVE_ONLY_LATEST = True  # @param {type:"boolean"} - 최신 체크포인트만 보관 (디스크 용량 절약)
SAVE_EVERY_WEIGHTS = True  # @param {type:"boolean"} - 체크포인트 저장 시 weights 폴더에 .pth 가출력 파일 저장
CUSTOM_PRETRAINED_G = ""  # @param {type:"string"} - 사용자 지정 G 모델 경로 (비워두면 기본 모델)
CUSTOM_PRETRAINED_D = ""  # @param {type:"string"} - 사용자 지정 D 모델 경로 (비워두면 기본 모델)
GPU_ID = "0"  # @param {type:"string"} - GPU 디바이스 ID

# Pretrained 경로 자동 판별
if CUSTOM_PRETRAINED_G != "":
    pretrained_g = CUSTOM_PRETRAINED_G
else:
    pretrained_g = f"assets/pretrained_v2/f0G{SAMPLE_RATE}.pth" if VERSION == "v2" else f"assets/pretrained/f0G{SAMPLE_RATE}.pth"

if CUSTOM_PRETRAINED_D != "":
    pretrained_d = CUSTOM_PRETRAINED_D
else:
    pretrained_d = f"assets/pretrained_v2/f0D{SAMPLE_RATE}.pth" if VERSION == "v2" else f"assets/pretrained/f0D{SAMPLE_RATE}.pth"

f0_flag = 1 if IF_F0 else 0
cache_flag = 1 if IF_CACHE_GPU else 0
latest_flag = 1 if SAVE_ONLY_LATEST else 0
weights_flag = 1 if SAVE_EVERY_WEIGHTS else 0

print(f"🚀 학습 시작! (Batch Size: {BATCH_SIZE}, Total Epochs: {TOTAL_EPOCH}, Save Every: {SAVE_EPOCH})")

# PyTorch 모델 학습 파이프라인 실행
!python train/train.py \
  -e {EXP_NAME} \
  -sr {SAMPLE_RATE} \
  -f0 {f0_flag} \
  -bs {BATCH_SIZE} \
  -g {GPU_ID} \
  -te {TOTAL_EPOCH} \
  -se {SAVE_EPOCH} \
  -pg {pretrained_g} \
  -pd {pretrained_d} \
  -l {latest_flag} \
  -c {cache_flag} \
  -sw {weights_flag} \
  -v {VERSION}

print("🎉 RVC 모델 학습 완료!")

---
## 8. FAISS Index (검색 인덱스) 생성
음조 변환 시 자연스러운 음색 복원을 위해 FAISS feature `.index` 파일을 생성합니다.

In [ ]:
OUTSIDE_INDEX_ROOT = "assets/indices"

# 인덱스 생성 실행
!python train/train_index.py {EXP_NAME} {VERSION} {OUTSIDE_INDEX_ROOT} {NUM_PROCESSES}

print("✅ FAISS Index 파일 생성 완료!")

---
## 9. 구글 드라이브로 결과물 내보내기
완성된 모델 파일(`.pth`)과 인덱스 파일(`.index`)을 내 Google Drive의 `RVC_Output` 폴더로 저장합니다.

In [ ]:
import shutil
import glob
import os

DRIVE_SAVE_PATH = "/content/drive/MyDrive/RVC_Output"  # @param {type:"string"}
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# 1. .pth weights 파일 내보내기
pth_files = glob.glob(f"assets/weights/{EXP_NAME}*.pth") + glob.glob(f"weights/{EXP_NAME}*.pth")
copied_pth = 0
for pth in pth_files:
    dst = os.path.join(DRIVE_SAVE_PATH, os.path.basename(pth))
    shutil.copy(pth, dst)
    print(f"📦 [Weights 백업] {pth} -> {dst}")
    copied_pth += 1

# 2. .index 파일 내보내기
index_files = glob.glob(f"logs/{EXP_NAME}/added_*.index") + glob.glob(f"assets/indices/*{EXP_NAME}*.index")
copied_idx = 0
for idx in set(index_files):
    dst = os.path.join(DRIVE_SAVE_PATH, os.path.basename(idx))
    shutil.copy(idx, dst)
    print(f"🔍 [Index 백업] {idx} -> {dst}")
    copied_idx += 1

print(f"\n🎉 구글 드라이브 저장 완료! (Weights: {copied_pth}개, Index: {copied_idx}개)")
print(f"📁 드라이브 경로: {DRIVE_SAVE_PATH}")